# MedCLIP-SAMv2 — Evaluation vs Ground Truth

Computes per-muscle metrics for MedCLIP-SAMv2 segmentations against the
myosegmenTUM ground truth, for both water and fat-fraction sequences.

| Parameter | Water | Fat Fraction |
|---|---|---|
| Seg dir | `segs_water/` | `segmentations_fat_frac/` |
| Suffix | `_medclipsamv2.npz` | `_medclipsamv2.npz` |
| Result dir | `results_water/` | `results_fat_frac/` |
| CSV suffix | `medclipsamv2_water` | `medclipsamv2_fatfrac` |

NPZ keys (string): `R_gracilis`, `L_gracilis`, `R_sartorius`, `L_sartorius`

In [1]:
import glob
import os
import re
import numpy as np
import pandas as pd
import SimpleITK as sitk
from dissector.evaluation import binary_cross_entropy, boundary_iou_3d, inter_slice_dice

In [2]:
BOUNDARY_DISTANCE = 1

EVAL_DIR = r'C:\Projects\dissector\eval_notebooks'
BASE_DIR = os.path.join(EVAL_DIR, 'medclipsamv2')
GT_BASE  = os.path.join(EVAL_DIR, 'myosegmenTUM')

SEG_DIR_WATER = os.path.join(BASE_DIR, 'segs_water')
SEG_DIR_FF    = os.path.join(BASE_DIR, 'segmentations_fat_frac')

# (muscle_name, gt_label_int, npz_key)
MUSCLES = [
    ('R_gracilis',  5, 'R_gracilis'),
    ('L_gracilis',  1, 'L_gracilis'),
    ('R_sartorius', 8, 'R_sartorius'),
    ('L_sartorius', 4, 'L_sartorius'),
]

print('GT_BASE  :', os.path.abspath(GT_BASE))
print('Water    :', SEG_DIR_WATER, '— exists:', os.path.isdir(SEG_DIR_WATER))
print('FF       :', SEG_DIR_FF,    '— exists:', os.path.isdir(SEG_DIR_FF))

GT_BASE  : C:\Projects\dissector\eval_notebooks\myosegmenTUM
Water    : C:\Projects\dissector\eval_notebooks\medclipsamv2\segs_water — exists: True
FF       : C:\Projects\dissector\eval_notebooks\medclipsamv2\segmentations_fat_frac — exists: True


In [3]:
def parse_subject_stack(filename, modality_tag):
    """
    Extract (subject, stack_num) from filenames like:
      HV001_1_WATER_stack1_medclipsamv2.npz
      P004_1_FATFRACTION_stack2_medclipsamv2.npz
    """
    pattern = rf'(.+)_{modality_tag}_stack(\d+)_medclipsamv2\.npz'
    m = re.match(pattern, os.path.basename(filename))
    if not m:
        return None, None
    return m.group(1), m.group(2)


def evaluate_muscle(muscle_name, gt_label_idx, npz_key,
                    seg_files, modality_tag, result_dir, csv_suffix):
    results = []
    for seg_file in seg_files:
        subject, stack_num = parse_subject_stack(seg_file, modality_tag)
        if subject is None:
            print(f'  could not parse: {os.path.basename(seg_file)}, skipping')
            continue

        gt_path = os.path.join(GT_BASE, subject, 'SegmentationMasks',
                               f'combined_gt_stack{stack_num}.mha')
        if not os.path.exists(gt_path):
            print(f'  GT not found: {gt_path}, skipping')
            continue

        gt_image = sitk.ReadImage(gt_path)
        gt       = sitk.Cast(gt_image == gt_label_idx, sitk.sitkUInt8)
        gt_arr   = sitk.GetArrayFromImage(gt).astype(float)

        data = np.load(seg_file)
        if npz_key not in data.files:
            print(f'  key "{npz_key}" not in {os.path.basename(seg_file)}, skipping')
            print(f'  available keys: {data.files}')
            continue
        pred_arr = data[npz_key].astype(np.uint8)

        pred_sitk = sitk.GetImageFromArray(pred_arr)
        pred_sitk.CopyInformation(gt_image)
        pred = sitk.Cast(pred_sitk, sitk.sitkUInt8)

        dice_filter = sitk.LabelOverlapMeasuresImageFilter()
        dice_filter.Execute(gt, pred)

        if gt_arr.sum() > 0 and pred_arr.sum() > 0:
            hd_filter = sitk.HausdorffDistanceImageFilter()
            hd_filter.Execute(gt, pred)
            hd = hd_filter.GetHausdorffDistance()
        else:
            print(f'  {os.path.basename(seg_file)}: empty mask '
                  f'(gt={int(gt_arr.sum())} pred={int(pred_arr.sum())}), HD=NaN')
            hd = np.nan

        results.append({
            'subject':                              subject,
            'stack':                                stack_num,
            'pred_file':                            os.path.basename(seg_file),
            'gt_path':                              gt_path,
            f'{muscle_name}_dice':                  dice_filter.GetDiceCoefficient(),
            f'{muscle_name}_hausdorff':             hd,
            f'{muscle_name}_jaccard':               dice_filter.GetJaccardCoefficient(),
            f'{muscle_name}_volume_similarity':     dice_filter.GetVolumeSimilarity(),
            f'{muscle_name}_false_negative':        dice_filter.GetFalseNegativeError(),
            f'{muscle_name}_false_positive':        dice_filter.GetFalsePositiveError(),
            f'{muscle_name}_bce':                   binary_cross_entropy(gt_arr, pred_arr.astype(float)),
            f'{muscle_name}_boundary_iou_3d':       boundary_iou_3d(BOUNDARY_DISTANCE, gt_arr, pred_arr.astype(float)),
            f'{muscle_name}_inter_slice_dice_pred': inter_slice_dice(pred_arr.astype(float)),
            f'{muscle_name}_inter_slice_dice_gt':   inter_slice_dice(gt_arr),
        })

    df = pd.DataFrame(results)
    os.makedirs(result_dir, exist_ok=True)
    csv_path = os.path.join(result_dir, f'df_{muscle_name}_{csv_suffix}.csv')
    df.to_csv(csv_path, index=False)
    print(f'  Saved {len(df)} rows -> {csv_path}')
    return df


print('Functions ready.')

Functions ready.


## Water

In [4]:
RESULT_DIR_WATER = os.path.join(BASE_DIR, 'results_water')

seg_files_water = sorted(glob.glob(os.path.join(SEG_DIR_WATER, '*_medclipsamv2.npz')))
print(f'Found {len(seg_files_water)} water files')
for f in seg_files_water[:5]:
    print(' ', os.path.basename(f))

Found 46 water files
  HV001_1_WATER_stack1_medclipsamv2.npz
  HV001_1_WATER_stack2_medclipsamv2.npz
  HV001_2_WATER_stack1_medclipsamv2.npz
  HV001_2_WATER_stack2_medclipsamv2.npz
  HV001_3_WATER_stack1_medclipsamv2.npz


In [ ]:
dfs_water = {}
for muscle_name, gt_idx, npz_key in MUSCLES:
    print(f'\n── {muscle_name} (gt={gt_idx}, key="{npz_key}") ──')
    dfs_water[muscle_name] = evaluate_muscle(
        muscle_name, gt_idx, npz_key,
        seg_files_water, 'WATER',
        RESULT_DIR_WATER, 'medclipsamv2_water',
    )
print('\nDone.')


── R_gracilis (gt=5, key="R_gracilis") ──


In [ ]:
for name, df in dfs_water.items():
    print(f'\n── {name} ──')
    display(df[['subject', 'stack', f'{name}_dice', f'{name}_hausdorff',
                f'{name}_jaccard', f'{name}_boundary_iou_3d']].head(10))

## Fat Fraction

In [ ]:
RESULT_DIR_FF = os.path.join(BASE_DIR, 'results_fat_frac')

seg_files_ff = sorted(glob.glob(os.path.join(SEG_DIR_FF, '*_medclipsamv2.npz')))
print(f'Found {len(seg_files_ff)} fat fraction files')
for f in seg_files_ff[:5]:
    print(' ', os.path.basename(f))

In [ ]:
dfs_ff = {}
for muscle_name, gt_idx, npz_key in MUSCLES:
    print(f'\n── {muscle_name} (gt={gt_idx}, key="{npz_key}") ──')
    dfs_ff[muscle_name] = evaluate_muscle(
        muscle_name, gt_idx, npz_key,
        seg_files_ff, 'FATFRACTION',
        RESULT_DIR_FF, 'medclipsamv2_fatfrac',
    )
print('\nDone.')

In [ ]:
for name, df in dfs_ff.items():
    print(f'\n── {name} ──')
    display(df[['subject', 'stack', f'{name}_dice', f'{name}_hausdorff',
                f'{name}_jaccard', f'{name}_boundary_iou_3d']].head(10))